# Session 3: Linear Programming für Produktions- und Absatzplanung

**Kurs:** Optimization and Machine Learning — FH Aachen, SS 2026  
**Datum:** 11.05.2026 | **Rolle:** Präsentation  

---

## Was ist das Ziel dieser Session?

Wir lösen ein klassisches **Produktionsplanungsproblem** mit Linear Programming (LP).

Stell dir vor: Ein Unternehmen produziert drei Produkte (A, B, C) und will möglichst viel **Gewinn** machen. Aber es gibt Grenzen:
- Zwei Maschinen haben begrenzte Kapazität (Maschinenstunden)
- Der Markt nimmt nur eine begrenzte Menge ab (Nachfrageobergrenze)

Die Frage lautet: **Wie viel von jedem Produkt sollen wir herstellen, um den Gewinn zu maximieren?**

Das ist genau die Art von Problem, die man mit LP elegant lösen kann.

---

## Die gegebenen Daten

| | Produkt A | Produkt B | Produkt C |
|---|---|---|---|
| **Gewinn pro Stück (€)** | 20 | 25 | 18 |
| **Maschinenstunden M1** | 2 | 3 | 1 |
| **Maschinenstunden M2** | 1 | 1 | 2 |
| **Max. Nachfrage** | 40 | 30 | 50 |

**Kapazitäten:**
- Maschine 1: max. 100 Stunden
- Maschine 2: max. 80 Stunden

**Variablen:** $x_A, x_B, x_C \geq 0$ — wie viel von jedem Produkt wird produziert.

---

## Das LP-Problem in Formeln

**Zielfunktion (maximieren):**
$$\max\; 20 x_A + 25 x_B + 18 x_C$$

**Constraints (Bedingungen):**

Maschine 1:  $2x_A + 3x_B + x_C \leq 100$  
Maschine 2:  $x_A + x_B + 2x_C \leq 80$  
Nachfrage A: $x_A \leq 40$  
Nachfrage B: $x_B \leq 30$  
Nachfrage C: $x_C \leq 50$  
Nichtneg.:   $x_A, x_B, x_C \geq 0$

**In Matrixform** (LP-Standardform: $A x \leq b$):

$$A = \begin{pmatrix} 2 & 3 & 1 \\ 1 & 1 & 2 \\ 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{pmatrix}, \quad b = \begin{pmatrix} 100 \\ 80 \\ 40 \\ 30 \\ 50 \end{pmatrix}$$

In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt

## Systemparameter

Hier definieren wir alle gegebenen Werte aus der Aufgabe als NumPy-Arrays.

In [ ]:
# Gewinn pro Stück für Produkt A, B, C
profit = np.array([20.0, 25.0, 18.0])

# Maschinenstunden pro Stück auf Maschine 1 und 2
a1 = np.array([2.0, 3.0, 1.0])  # Maschine 1: 2h für A, 3h für B, 1h für C
a2 = np.array([1.0, 1.0, 2.0])  # Maschine 2: 1h für A, 1h für B, 2h für C

# Kapazitäten der Maschinen (Stunden)
c1 = 100.0  # Maschine 1
c2 = 80.0   # Maschine 2

# Maximale Nachfrage (wie viel der Markt maximal abnimmt)
d_max = np.array([40.0, 30.0, 50.0])

# Anzahl Produkte
n = 3

print("Gewinn pro Stück:", profit)
print("Maschine 1 Stunden/Stück:", a1)
print("Maschine 2 Stunden/Stück:", a2)
print("Kapazität M1:", c1, "| M2:", c2)
print("Max. Nachfrage:", d_max)

---

## Task 1: Basic Production Planning Model

**Aufgabe (wortwörtlich aus dem Übungsblatt):**

> Formulate and implement the profit maximization problem as a linear program.  
> Your code must:
> - define the production variables,
> - maximize total profit,
> - enforce both machine capacity constraints,
> - enforce all demand upper bounds,
> - solve the model and print the optimal plan.

---

### Was wird gefragt?

Wir sollen das LP-Modell als Code umsetzen und lösen. Das heißt:

1. **Variablen definieren:** Wie viel produzieren wir von A, B und C? Das sind die Unbekannten, die cvxpy für uns findet.
2. **Zielfunktion formulieren:** Maximiere den Gesamtgewinn = Summe von (Gewinn × Menge) über alle Produkte.
3. **Constraints formulieren:** Maschinen nicht überlasten + Nachfrage nicht überschreiten.
4. **Lösen und ausgeben.**

**Wichtig:** cvxpy minimiert immer. Um zu maximieren, schreiben wir: `cp.Maximize(...)` — cvxpy macht intern dann `-1 × Zielfunktion`.

In [ ]:
# Entscheidungsvariablen: x[0]=A, x[1]=B, x[2]=C, alle >= 0 (non-negative)
x = cp.Variable(n, nonneg=True)

# Zielfunktion: Gesamtgewinn = profit · x (Skalarprodukt)
objective = cp.Maximize(profit @ x)

# Constraints (Bedingungen)
constraints = [
    a1 @ x <= c1,      # Maschine 1 darf nicht mehr als 100h laufen
    a2 @ x <= c2,      # Maschine 2 darf nicht mehr als 80h laufen
    x <= d_max,        # Keine Überproduktion über die Nachfrage hinaus
]

# Problem aufstellen und lösen
prob1 = cp.Problem(objective, constraints)
prob1.solve()

x_opt = x.value
print(f"Status:          {prob1.status}")
print(f"Optimaler Plan:  A={x_opt[0]:.2f}, B={x_opt[1]:.2f}, C={x_opt[2]:.2f}")
print(f"Maximaler Gewinn: {prob1.value:.2f} €")

---

## Task 2: Structured Output and Verification

**Aufgabe (wortwörtlich aus dem Übungsblatt):**

> Extend your code so that it outputs:
> - optimal production quantities,
> - total profit,
> - used and unused capacity on each machine,
> - which demand bounds are active,
> - whether all constraints are satisfied up to numerical tolerance.
>
> Present the results in a clear table or dictionary style output.

---

### Was wird gefragt?

Wir sollen die Lösung ausführlich analysieren und strukturiert ausgeben:

- **Kapazitätsauslastung:** Wie viele Stunden verbraucht der optimale Plan auf M1 und M2? Wie viel bleibt ungenutzt?
- **Aktive Constraints:** Welche Nachfreobergrenzen werden genau erreicht ("aktiv")? Das sind besonders wichtige Engpässe — wenn man hier mehr Kapazität hätte, würde der Gewinn steigen.
- **Plausibilitätsprüfung:** Sind wirklich alle Bedingungen erfüllt? (Numerische Lösungen haben immer kleine Rundungsfehler, deswegen `1e-5` Toleranz.)

**Was ist ein aktiver Constraint?**  
Ein Constraint ist *aktiv*, wenn er exakt ausgeschöpft ist — z.B. wenn Maschine 1 genau 100 Stunden läuft (kein Spielraum übrig). Aktive Constraints bestimmen, was uns wirklich limitiert.

In [ ]:
# Kapazitätsverbrauch berechnen
used_m1 = a1 @ x_opt          # Tatsächlich verbrauchte Stunden auf M1
used_m2 = a2 @ x_opt          # Tatsächlich verbrauchte Stunden auf M2
slack_m1 = c1 - used_m1       # Ungenutzte Stunden auf M1
slack_m2 = c2 - used_m2       # Ungenutzte Stunden auf M2

print("=" * 45)
print("OPTIMALER PRODUKTIONSPLAN")
print("=" * 45)
print(f"{'Produkt':<12} {'Menge':>8} {'Max.Nachfr.':>12} {'Aktiv?':>8}")
print("-" * 45)
for i, name in enumerate(['A', 'B', 'C']):
    active = abs(x_opt[i] - d_max[i]) < 1e-5   # Nachfrage genau ausgeschöpft?
    print(f"Produkt {name:<4} {x_opt[i]:>8.2f} {d_max[i]:>12.1f} {'✓' if active else '—':>8}")

print("-" * 45)
print(f"Gesamtgewinn: {prob1.value:.2f} €")

print("\n" + "=" * 45)
print("MASCHINENAUSLASTUNG")
print("=" * 45)
print(f"{'Maschine':<12} {'Genutzt':>8} {'Kapazität':>10} {'Rest':>8} {'Aktiv?':>8}")
print("-" * 45)
print(f"{'M1':<12} {used_m1:>8.2f} {c1:>10.1f} {slack_m1:>8.2f} {'✓' if slack_m1 < 1e-5 else '—':>8}")
print(f"{'M2':<12} {used_m2:>8.2f} {c2:>10.1f} {slack_m2:>8.2f} {'✓' if slack_m2 < 1e-5 else '—':>8}")

print("\n" + "=" * 45)
print("PLAUSIBILITÄTSPRÜFUNG")
print("=" * 45)
tol = 1e-5
checks = [
    ("M1 Kapazität",   a1 @ x_opt <= c1 + tol),
    ("M2 Kapazität",   a2 @ x_opt <= c2 + tol),
    ("Nachfrage A",    x_opt[0] <= d_max[0] + tol),
    ("Nachfrage B",    x_opt[1] <= d_max[1] + tol),
    ("Nachfrage C",    x_opt[2] <= d_max[2] + tol),
    ("Nichtnegativität", np.all(x_opt >= -tol)),
]
for name, ok in checks:
    print(f"  {name:<20} {'✓ OK' if ok else '✗ VERLETZT'}")
print("Alle Constraints erfüllt:", all(ok for _, ok in checks))

---

## Task 3: Scenario Analysis

**Aufgabe (wortwörtlich aus dem Übungsblatt):**

> Study the effect of changing one model component at a time.  
> Implement at least the following three scenarios:
> 1. Increase the profit of product C from 18 to 30.
> 2. Increase machine 2 capacity from 80 to 110.
> 3. Reduce demand of product B from 30 to 10.
>
> For each scenario:
> - solve the modified model,
> - compare the new production plan with the baseline,
> - compare the objective value with the baseline,
> - summarize which constraint or parameter change has the strongest effect.

---

### Was wird gefragt?

Wir verändern jeweils **einen Parameter** und schauen, was passiert:

- **Szenario 1:** Produkt C wird lukrativer (Gewinn 18 → 30). Wird C jetzt mehr produziert?
- **Szenario 2:** Maschine 2 bekommt mehr Kapazität (80 → 110 Stunden). Kann die Fabrik mehr produzieren?
- **Szenario 3:** Der Markt nimmt weniger B ab (Nachfrage 30 → 10). Müssen wir den Plan anpassen?

**Warum ist das wichtig?**  
In der Realität ändern sich Preise, Kapazitäten und Nachfragen ständig. Mit Szenarioanalysen sieht man, wie robust der Plan ist und wo die Engpässe liegen.

**Hilfsfunktion:** Wir schreiben eine Funktion, die das LP für beliebige Parameter löst — das ist sauberer als Code dreimal zu kopieren.

In [ ]:
def solve_production(profit_v, a1_v, a2_v, c1_v, c2_v, d_max_v):
    """Löst das Produktionsplanungs-LP für gegebene Parameter."""
    n_v = len(profit_v)
    x_v = cp.Variable(n_v, nonneg=True)
    prob_v = cp.Problem(
        cp.Maximize(profit_v @ x_v),
        [a1_v @ x_v <= c1_v, a2_v @ x_v <= c2_v, x_v <= d_max_v]
    )
    prob_v.solve()
    return prob_v.status, prob_v.value, x_v.value


# Baseline (Ausgangsszenario)
baseline_val = prob1.value
baseline_x   = x_opt.copy()

# Szenario 1: Gewinn C von 18 auf 30
profit_s1 = profit.copy(); profit_s1[2] = 30.0
_, val_s1, x_s1 = solve_production(profit_s1, a1, a2, c1, c2, d_max)

# Szenario 2: M2-Kapazität von 80 auf 110
_, val_s2, x_s2 = solve_production(profit, a1, a2, c1, 110.0, d_max)

# Szenario 3: Nachfrage B von 30 auf 10
d_max_s3 = d_max.copy(); d_max_s3[1] = 10.0
_, val_s3, x_s3 = solve_production(profit, a1, a2, c1, c2, d_max_s3)

# Vergleich ausgeben
print(f"{'Szenario':<30} {'A':>6} {'B':>6} {'C':>6} {'Gewinn':>10} {'Δ Gewinn':>10}")
print("-" * 72)
print(f"{'Baseline':<30} {baseline_x[0]:>6.1f} {baseline_x[1]:>6.1f} {baseline_x[2]:>6.1f} {baseline_val:>10.2f} {'—':>10}")
for name, x_s, val_s in [
    ("S1: Gewinn C → 30", x_s1, val_s1),
    ("S2: M2-Kap. → 110", x_s2, val_s2),
    ("S3: Nachfr. B → 10", x_s3, val_s3),
]:
    delta = val_s - baseline_val
    print(f"{name:<30} {x_s[0]:>6.1f} {x_s[1]:>6.1f} {x_s[2]:>6.1f} {val_s:>10.2f} {delta:>+10.2f}")

print("\n→ Stärkstes Szenario:", max([
    ("S1", val_s1 - baseline_val),
    ("S2", val_s2 - baseline_val),
    ("S3", val_s3 - baseline_val),
], key=lambda t: t[1]))

---

## Task 4: Multi Period Extension

**Aufgabe (wortwörtlich aus dem Übungsblatt):**

> Extend the model to two production periods $t = 1, 2$. Let $x_{i,t}$ denote the production quantity of product $i$ in period $t$.
> Use:
> - the same per unit profits in both periods,
> - machine capacities of 100 and 80 in each period,
> - period specific demands
> $$d^{(1)} = (20, 15, 30)^T, \quad d^{(2)} = (25, 20, 35)^T$$
> - inventory variables $s_{i,t} \geq 0$,
> - inventory holding cost 1 per unit and period.
>
> Model a two period production planning problem with inventory carryover.  
> Your implementation must:
> - include production and inventory variables,
> - enforce inventory balance equations,
> - maximize total profit minus holding costs,
> - report the optimal production and storage plan.

---

### Was wird gefragt?

Jetzt erweitern wir das Modell auf **zwei Zeitperioden** (z.B. zwei Wochen). Das ist realistischer:

- In Periode 1 produzieren wir mehr als verkauft werden muss → Rest lagern wir ein.
- In Periode 2 nutzen wir das Lager, statt alles neu zu produzieren.
- **Lagerhaltungskosten:** Jede gelagerte Einheit kostet 1 € pro Periode.

**Die Inventarbilanz-Gleichung** (Kernidee):
$$s_{i,t} = s_{i,t-1} + x_{i,t} - \text{Nachfrage}_{i,t}$$

Also: Lagerbestand am Ende von Periode $t$ = Lagerbestand vorher + neu produziert - verkauft

Anfangs: kein Lagerbestand ($s_{i,0} = 0$).

**Variablen:**
- $x_{i,t}$: Produktion von Produkt $i$ in Periode $t$ — als 3×2-Matrix
- $s_{i,t}$: Lagerbestand von Produkt $i$ nach Periode $t$ — als 3×2-Matrix

In [ ]:
# Periodenspezifische Nachfragen (Zeilen=Produkte, Spalten=Perioden)
d_t1 = np.array([20.0, 15.0, 30.0])  # Nachfrage Periode 1
d_t2 = np.array([25.0, 20.0, 35.0])  # Nachfrage Periode 2
T = 2  # Anzahl Perioden

# Variablen: x[i,t] = Produktion, s[i,t] = Lagerbestand
X = cp.Variable((n, T), nonneg=True)  # Produktionsmengen (3×2 Matrix)
S = cp.Variable((n, T), nonneg=True)  # Lagerbestände (3×2 Matrix)

# Gesamtgewinn über beide Perioden (Gewinn × Nachfrage, da alles verkauft wird)
total_profit = profit @ d_t1 + profit @ d_t2

# Lagerhaltungskosten: 1 € pro gelagerter Einheit pro Periode
holding_cost = cp.sum(S)  # Summe aller Lagerbestände × 1

# Zielfunktion: Gewinn - Lagerkosten
# Gewinn entsteht aus dem Verkauf (= Nachfrage), da wir genau die Nachfrage decken wollen
# Wir maximieren: Gewinn aus Produktion minus Lagerkosten
revenue = profit @ X[:, 0] + profit @ X[:, 1]  # Umsatz aus Produktion
obj4 = cp.Maximize(revenue - holding_cost)

constraints4 = []

# Maschinenkapazität für jede Periode einhalten
for t in range(T):
    constraints4.append(a1 @ X[:, t] <= c1)
    constraints4.append(a2 @ X[:, t] <= c2)

# Inventarbilanz-Gleichungen
# Periode 1: Lager am Ende = Produktion - Nachfrage (kein Anfangslager)
constraints4.append(S[:, 0] == X[:, 0] - d_t1)

# Periode 2: Lager am Ende = Lager aus P1 + Produktion - Nachfrage
constraints4.append(S[:, 1] == S[:, 0] + X[:, 1] - d_t2)

# Nachfrage muss in jeder Periode gedeckt sein (nicht mehr produzieren als verkauft + gelagert)
constraints4.append(X[:, 0] >= d_t1)  # Mindestens Nachfrage Periode 1 produzieren

prob4 = cp.Problem(obj4, constraints4)
prob4.solve()

print(f"Status: {prob4.status}")
print(f"\nOptimaler Wert (Gewinn - Lagerkosten): {prob4.value:.2f} €")
print("\nProduktionsplan X (Zeilen=A,B,C | Spalten=Periode 1,2):")
print(np.round(X.value, 2))
print("\nLagerbestand S (Zeilen=A,B,C | Spalten=nach P1,nach P2):")
print(np.round(S.value, 2))

---

## Task 5: Automatic Experiment Framework

**Aufgabe (wortwörtlich aus dem Übungsblatt):**

> Write a small experiment framework in Python that:
> - takes profits, capacities, and demands as input,
> - solves the corresponding production planning problem,
> - returns the optimal plan and the optimal value,
> - allows repeated evaluation of many scenarios with little manual modification.
>
> Use this framework to test at least five additional random or manually designed scenarios and summarize the results.

---

### Was wird gefragt?

Wir bauen ein kleines **Framework** (= wiederverwendbare Struktur), mit dem man ganz einfach neue Szenarien testen kann, ohne Code zu duplizieren.

Dann testen wir mindestens 5 verschiedene Kombinationen von Parametern und fassen die Ergebnisse zusammen.

**Warum ist das praktisch?**  
In echten Projekten möchte man schnell viele Varianten testen (z.B. verschiedene Marktlagen, Preisänderungen). Ein Framework macht das systematisch und fehlerrobust.

In [ ]:
def run_experiment(name, profit_v, c1_v, c2_v, d_max_v,
                   a1_v=None, a2_v=None):
    """Framework: Löst ein Produktionsplanungs-LP und gibt Ergebnis zurück."""
    if a1_v is None: a1_v = a1
    if a2_v is None: a2_v = a2
    status, val, plan = solve_production(profit_v, a1_v, a2_v, c1_v, c2_v, d_max_v)
    return {
        "name": name,
        "status": status,
        "profit": val,
        "plan_A": plan[0] if plan is not None else None,
        "plan_B": plan[1] if plan is not None else None,
        "plan_C": plan[2] if plan is not None else None,
    }


# 5+ Szenarien definieren und testen
experiments = [
    # Baseline
    run_experiment("Baseline",          profit,                   c1,   c2,   d_max),
    # Hohe Gewinne für alle
    run_experiment("Alle Gewinne ×2",   profit * 2,               c1,   c2,   d_max),
    # Sehr knappe M1-Kapazität
    run_experiment("M1-Kap. = 50",      profit,                   50.0, c2,   d_max),
    # Sehr knappe M2-Kapazität
    run_experiment("M2-Kap. = 30",      profit,                   c1,   30.0, d_max),
    # Doppelte Nachfrage
    run_experiment("Nachfrage ×2",       profit,                   c1,   c2,   d_max * 2),
    # Nur Produkt A rentabel
    run_experiment("Nur A rentabel",     np.array([20., 1., 1.]), c1,   c2,   d_max),
    # Alle Maschinen voll
    run_experiment("M1+M2 voll (150/120)", profit,               150., 120., d_max),
]

# Ergebnisse als Tabelle
print(f"{'Szenario':<25} {'Status':<12} {'A':>6} {'B':>6} {'C':>6} {'Gewinn':>10}")
print("-" * 68)
for e in experiments:
    if e['status'] == 'optimal':
        print(f"{e['name']:<25} {e['status']:<12} {e['plan_A']:>6.1f} {e['plan_B']:>6.1f} {e['plan_C']:>6.1f} {e['profit']:>10.2f}")
    else:
        print(f"{e['name']:<25} {e['status']:<12} {'—':>6} {'—':>6} {'—':>6} {'—':>10}")

---

## Mögliche Challenger-Fragen (für die Präsentationsgruppe)

Da wir bei Session 3 **präsentieren**, sind diese Fragen zum Üben — aber umgekehrt könnten andere Gruppen uns ähnliches fragen.

---

### Zu Task 1 & 2

**F1: Warum verwendet ihr `cp.Maximize` und nicht `cp.Minimize`?**  
→ Weil wir Gewinn *maximieren* wollen. cvxpy unterstützt beides. Intern wird `Maximize(f)` zu `Minimize(-f)` umgeformt, was der LP-Standardform entspricht.

**F2: Was bedeutet es, wenn ein Constraint "aktiv" ist?**  
→ Der Constraint ist exakt ausgeschöpft (kein Spielraum übrig). Aktive Constraints sind die Engpässe — bei M1 aktiv: mehr Maschinenstunden würden mehr Gewinn bringen. Inaktive Constraints könnten entfernt werden, ohne die Lösung zu ändern.

**F3: Was passiert, wenn wir den Nachfrageconstraint weglassen?**  
→ Das LP ist immer noch bounded (durch Maschinenkapazitäten), aber wir würden unrealistische Mengen produzieren. Z.B. könnte der Solver Produkt B auf 33 Stück setzen, obwohl der Markt nur 30 abnimmt.

---

### Zu Task 3

**F4: Welches Szenario hat den stärksten Effekt auf den Gewinn? Warum?**  
→ Hängt vom Ergebnis ab. Typischerweise hat eine Kapazitätserweiterung den größten Effekt, wenn Maschinen aktive Constraints sind. Falls Nachfrage der Engpass ist, hilft mehr Kapazität wenig.

**F5: Was versteht man unter "Sensitivitätsanalyse"?**  
→ Untersuchen, wie empfindlich die Lösung auf kleine Parameteränderungen reagiert. Wenn eine kleine Preisänderung den Plan komplett ändert, ist die Lösung instabil. Task 3 ist eine manuelle Sensitivitätsanalyse.

---

### Zu Task 4

**F6: Was ist die Inventarbilanz-Gleichung und warum braucht man sie?**  
→ Sie stellt sicher, dass der Lagerbestand konsistent ist: $s_{i,t} = s_{i,t-1} + x_{i,t} - d_{i,t}$. Ohne sie könnte das Modell Waren "aus dem Nichts" verkaufen.

**F7: Warum entstehen Lagerhaltungskosten? Wie beeinflussen sie die Lösung?**  
→ Lager kostet Geld (Platz, Kapital). Die Kosten incentivieren den Solver, möglichst wenig zu lagern. Ohne Kosten würde der Solver in P1 alles produzieren, was maximal möglich ist — unabhängig vom tatsächlichen Bedarf.

**F8: Was bedeutet $s_{i,2} \geq 0$ am Ende? Könnte man auch Endlagerbestand erzwingen?**  
→ Ja, man könnte $s_{i,2} = 0$ fordern (kein Restlager am Ende). Das ist realistisch, wenn der Planungshorizont abgeschlossen ist. Als Gleichungsconstraint würde es die Lösung weiter einschränken.

---

### Zu Task 5

**F9: Warum ist ein Framework besser als copy-paste?**  
→ Kein Doppelcode, weniger Fehler, leicht erweiterbar. Wenn sich der Kern des Problems ändert (z.B. Maschinen hinzufügen), muss man nur eine Stelle ändern, nicht jedes Szenario.

**F10: Wie würdet ihr das Framework erweitern, wenn es eine dritte Maschine gibt?**  
→ Parameter `a3` und `c3` hinzufügen, einen weiteren Constraint `a3 @ x <= c3` in die Funktion einbauen. Das ist der Vorteil von parametrisierten Funktionen.